<a href="https://colab.research.google.com/github/mattiademartino/Space_Data_2/blob/main/cursed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install cdsapi xarray netCDF4 scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 75.9 MB/s eta 0:00:00


In [2]:
import os

URL = 'https://cds.climate.copernicus.eu/api'
KEY = '511de32f-ddc9-4279-9914-283348312536'

# Create .cdsapirc file for cdsapi
with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
    f.write(f'url: {URL}\n')
    f.write(f'key: {KEY}\n')

print('CDSAPI credentials have been set up.')

CDSAPI credentials have been set up.


In [3]:
import cdsapi

URL = 'https://cds.climate.copernicus.eu/api'
KEY = '511de32f-ddc9-4279-9914-283348312536'

c = cdsapi.Client(url=URL, key=KEY)

In [4]:
script_content = """
import cdsapi

def main():
    print('Running ERA5 profile script...')
    # Add your specific script logic here

if __name__ == '__main__':
    main()
"""

with open('era5_profile.py', 'w') as f:
    f.write(script_content)

print('File era5_profile.py has been created.')

File era5_profile.py has been created.


In [5]:
!python era5_profile.py

Running ERA5 profile script...


In [6]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from scipy.interpolate import interp1d
import os, warnings
warnings.filterwarnings('ignore')


In [7]:
# ============================================================================
# CONFIGURATION
# ============================================================================

from pathlib import Path

# Drop location
LAT       = 47.40    # Dübendorf
LON       = 8.62
ELEV_ASL  = 448      # m ASL (Dübendorf airport)

# Drop time (UTC)
DROP_YEAR  = '2026'
DROP_MONTH = '02'
DROP_DAY   = '06'
DROP_HOURS = ['11:00']

# Canonical repo paths
DATA_DIR    = Path('./data/external_dataset')
OUTPUT_DIR  = Path('./figures/external_dataset')
NC_FILE     = DATA_DIR / 'era5_dubendorf_20260206.nc'
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Pressure levels covering 0–1500 m AGL above Dübendorf (448 m ASL = ~955 hPa)
PRESSURE_LEVELS = ['1000', '950', '900', '850']

# Physical constants
Rd = 287.05; g = 9.80665

def baro(p, p0): return 44330*(1-(p/p0)**(1/5.255))
def isa_T(h): return 288.15 - 0.0065*h
def isa_p(h): return 1013.25*(isa_T(h)/288.15)**5.255

print(f'ERA5 NetCDF will be stored in: {NC_FILE.resolve()}')
print(f'ERA5 figures will be stored in: {OUTPUT_DIR.resolve()}')

In [ ]:
# ============================================================================
# STEP 1: DOWNLOAD ERA5 (skipped if file already exists)
# ============================================================================

if os.path.exists(NC_FILE):
    print(f"ERA5 file already exists: {NC_FILE} — skipping download")
else:
    print("Downloading ERA5 data from Copernicus CDS...")
    print("(This requires ~/.cdsapirc with your API credentials)")

    try:
        # 'c' is already initialized in cell f88a66ba with URL and KEY
        c.retrieve(
            'reanalysis-era5-pressure-levels',
            {
                'product_type': 'reanalysis',
                'variable': [
                    'temperature',
                    'geopotential',
                    'relative_humidity',
                    'u_component_of_wind',
                    'v_component_of_wind',
                    # Removed 'specific_humidity' for a lighter download
                ],
                'pressure_level': PRESSURE_LEVELS,
                'year':  DROP_YEAR,
                'month': DROP_MONTH,
                'day':   DROP_DAY,
                'time':  DROP_HOURS,
                # Small bounding box around Dübendorf (lat_N, lon_W, lat_S, lon_E)
                'area':  [47.6, 8.4, 47.2, 8.8],
                'format': 'netcdf',
            },
            str(NC_FILE)
        )
        if os.path.exists(NC_FILE):
            print(f"  ✓ Downloaded: {NC_FILE}")
        else:
            print(f"  ✗ Download attempt finished, but file '{NC_FILE}' not found. It might have failed or been corrupted.")
            NC_FILE = None

    except ImportError:
        print("  ✗ cdsapi not installed. Run: pip install cdsapi")
        print("  → Continuing without ERA5 (only ISA + Payerne will be shown)")
        NC_FILE = None

    except Exception as e:
        print(f"  ✗ Download failed: {e}")
        print("  → Check ~/.cdsapirc credentials or CDS website for request status.")
        NC_FILE = None

(This requires ~/.cdsapirc with your API credentials)


2026-04-22 18:39:08,198 INFO Request ID is ad8ee511-7717-4299-988f-b94a4cd6cd8d
INFO:ecmwf.datastores.legacy_client:Request ID is ad8ee511-7717-4299-988f-b94a4cd6cd8d
2026-04-22 18:39:08,344 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted


In [ ]:


# ============================================================================
# STEP 2: LOAD AND PROCESS ERA5
# ============================================================================

era5_ok = False
era5_profiles = {}

if NC_FILE and os.path.exists(NC_FILE):
    print(f"\nProcessing ERA5 data...")
    ds = xr.open_dataset(NC_FILE)

    print(f"  Variables: {list(ds.data_vars)}")
    print(f"  Pressure levels: {ds.level.values} hPa")
    print(f"  Times: {ds.time.values}")

    # Spatial mean over the small bounding box (they're all very close)
    ds_loc = ds.mean(dim=['latitude','longitude'])

    # Loop over available hours → pick closest to drop time
    target_hour = np.datetime64(f'{DROP_YEAR}-{DROP_MONTH}-{DROP_DAY}T11:00')
    time_diffs  = np.abs(ds.time.values - target_hour)
    best_time   = ds.time.values[np.argmin(time_diffs)]
    ds_best     = ds_loc.sel(time=best_time)

    print(f"  Selected time: {best_time}")

    # Extract profiles
    p_lev  = ds_best.level.values           # hPa
    T_era5 = ds_best['t'].values - 273.15   # °C
    z_era5 = ds_best['z'].values / g        # geopotential → m ASL
    rh_era5 = ds_best['r'].values           # %
    u_era5  = ds_best['u'].values           # m/s eastward
    v_era5  = ds_best['v'].values           # m/s northward
    wspd_era5 = np.sqrt(u_era5**2 + v_era5**2)

    # Convert to AGL relative to Dübendorf
    h_era5_agl = z_era5 - ELEV_ASL

    # Sort by altitude (ascending)
    sort_idx = np.argsort(h_era5_agl)
    h_era5_agl = h_era5_agl[sort_idx]
    T_era5     = T_era5[sort_idx]
    p_lev_s    = p_lev[sort_idx]
    rh_era5    = rh_era5[sort_idx]
    wspd_era5  = wspd_era5[sort_idx]
    u_era5     = u_era5[sort_idx]
    v_era5     = v_era5[sort_idx]

    # Keep only levels above ground and below 1500 m AGL
    mask = (h_era5_agl >= -50) & (h_era5_agl <= 1500)
    h_era5_agl = h_era5_agl[mask]
    T_era5     = T_era5[mask]
    p_era5_hPa = p_lev_s[mask]
    rh_era5    = rh_era5[mask]
    wspd_era5  = wspd_era5[mask]
    u_era5     = u_era5[mask]
    v_era5     = v_era5[mask]

    era5_profiles = {
        'h_agl': h_era5_agl, 'T': T_era5, 'p': p_era5_hPa,
        'rh': rh_era5, 'wspd': wspd_era5, 'u': u_era5, 'v': v_era5,
        'time': str(best_time)
    }
    era5_ok = True

    print(f"  ✓ ERA5 profile: {len(h_era5_agl)} levels, "
          f"h = {h_era5_agl.min():.0f}–{h_era5_agl.max():.0f} m AGL")
    print(f"  T range: {T_era5.min():.1f}–{T_era5.max():.1f}°C")
    print(f"  Wind: {wspd_era5.min():.1f}–{wspd_era5.max():.1f} m/s")

NameError: name 'NC_FILE' is not defined

In [ ]:
# ============================================================================
# STEP 3: LOAD VAMOS DROP DATA (for comparison)
# ============================================================================

print("\nLoading VAMOS drop data...")

def clean(df):
    for c in df.columns: df[c] = pd.to_numeric(df[c], errors='coerce')
    return df.dropna().reset_index(drop=True)

sv   = clean(pd.read_csv(f"{DATA_DIR}/science_VAMOS.csv"))
t_v  = sv["timestamp_ms"].values/1000
p_v  = sv["pressure_hPa"].values
T_v  = sv["temperature_C"].values
p0_v = (np.median(p_v[:100])+np.median(p_v[-500:]))/2
h_v  = baro(p_v, p0_v)

p_sm = pd.Series(p_v).rolling(5,center=True,min_periods=1).median().values
dpdt = np.gradient(p_sm)/np.gradient(t_v)
idx  = np.where(dpdt>0.1)[0]
gaps = np.where(np.diff(idx)>15)[0]
S    = np.concatenate([[idx[0]],idx[gaps+1]]) if len(gaps) else np.array([idx[0]])
E    = np.concatenate([idx[gaps],[idx[-1]]]) if len(gaps) else np.array([idx[-1]])
ib   = np.argmax([p_v[e]-p_v[s] for s,e in zip(S,E)])
ds_v, de_v = S[ib], E[ib]
apex = max(0,ds_v-60)+int(np.argmin(p_v[max(0,ds_v-60):ds_v+1]))
post = np.where(p_v[de_v:min(len(p_v),de_v+60)]>p_v[apex]+30)[0]
land = de_v+(post[0] if len(post) else 0)
ta_v = t_v[apex]; tl_v = t_v[land]
dm_v = (t_v>=ta_v)&(t_v<=tl_v)

T_vd = T_v[dm_v]; h_vd = h_v[dm_v]
print(f"  Drop: {dm_v.sum()} pts, h={h_vd.min():.0f}–{h_vd.max():.0f} m AGL")


In [ ]:


# ============================================================================
# STEP 4: PAYERNE RADIOSONDE (hardcoded, same as before)
# ============================================================================

PAY_RAW = """935.9 491 1.9 0.9 93 4 0.3
933.8 509 1.1 1.0 99 45 0.9
930.4 538 0.8 0.7 99 40 1.1
927.8 561 0.6 0.5 100 29 1.2
925.2 584 0.4 0.4 100 19 1.2
922.6 606 0.3 0.3 100 12 1.3
919.1 637 0.1 0.1 100 7 1.3
916.5 660 0.0 0.0 100 9 1.2
914.9 674 -0.0 -0.0 100 12 1.1
912.3 696 -0.2 -0.2 100 23 0.8
909.4 722 -0.3 -0.3 100 50 0.5
905.6 755 -0.5 -0.5 100 123 0.5
902.3 784 -0.7 -0.8 100 151 0.9
900.5 801 -0.2 -2.6 84 159 1.1
898.3 821 1.0 -2.8 76 168 1.3
896.1 840 1.8 -3.4 68 176 1.6
893.3 865 2.5 -2.8 68 185 2.0
890.4 892 3.0 -2.5 67 192 2.6
887.0 923 3.1 -2.5 67 197 3.2
884.1 949 2.9 -2.8 66 201 3.9
881.0 978 2.8 -3.2 65 204 4.4
878.7 998 2.6 -3.8 62 206 4.7
875.0 1032 2.5 -4.1 62 209 5.2
872.5 1056 2.5 -5.2 57 210 5.4
869.2 1087 2.8 -5.6 54 211 5.7
866.9 1108 3.0 -5.1 55 210 5.8
864.5 1130 3.5 -4.0 58 210 5.8
862.3 1151 3.4 -3.3 62 209 5.9
859.8 1175 3.2 -3.2 63 209 6.0
857.6 1196 3.0 -3.1 64 211 6.2
855.5 1215 2.8 -2.6 67 212 6.6
853.2 1237 2.7 -2.5 69 215 7.0
851.0 1258 2.6 -2.5 69 217 7.5
849.3 1274 2.4 -2.4 70 219 7.8
847.1 1295 2.3 -2.1 72 220 8.3
844.8 1317 2.2 -1.9 74 221 8.6
842.6 1338 2.2 -2.0 74 222 8.9
840.4 1359 2.2 -2.3 72 223 9.2
838.1 1382 2.2 -2.4 72 223 9.4
835.9 1403 2.1 -2.4 72 224 9.5
833.7 1424 1.9 -2.5 73 224 9.7
831.5 1445 1.7 -2.5 74 223 9.9
829.2 1467 1.5 -2.5 75 222 10.0
827.0 1489 1.3 -2.5 76 220 10.1
825.9 1499 1.2 -2.5 76 219 10.1"""

_r = [[float(x) for x in ln.split()] for ln in PAY_RAW.strip().split('\n')]
pay = pd.DataFrame(_r, columns=['p','h_asl','T','Td','rh','wdir','wspd'])
pay['h_agl'] = pay['h_asl'] - ELEV_ASL
wr = np.deg2rad(pay['wdir'])
pay['u'] = -pay['wspd']*np.sin(wr)
pay['v'] = -pay['wspd']*np.cos(wr)


In [ ]:


# ============================================================================
# STEP 5: PLOT
# ============================================================================

print("\nGenerating vertical profile comparison figure...")

C_VAMOS = '#1f77b4'
C_ERA5  = '#e377c2'   # pink — distinct from everything else
C_PAY   = '#ff7f0e'
C_ISA   = 'black'

h_isa_agl = np.linspace(0, 1100, 300)
T_isa_C   = isa_T(h_isa_agl + ELEV_ASL) - 273.15
p_isa     = isa_p(h_isa_agl + ELEV_ASL)

fig = plt.figure(figsize=(16, 7))
gs  = gridspec.GridSpec(1, 4, figure=fig, wspace=0.30)

H_LIM = (0, 900)

# ── Panel 1: Temperature ──────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
ax1.plot(T_isa_C, h_isa_agl, 'k--', lw=2, label='ISA', zorder=1)
ax1.plot(pay['T'], pay['h_agl'], color=C_PAY, lw=2.5, marker='o', ms=3,
         label='Payerne\n(140 km SW)', zorder=2)
if era5_ok:
    ax1.plot(era5_profiles['T'], era5_profiles['h_agl'],
             color=C_ERA5, lw=2.5, marker='s', ms=6,
             label=f'ERA5\n(Dübendorf, {era5_profiles["time"][:13]})', zorder=3)
ax1.scatter(T_vd, h_vd, s=12, c=C_VAMOS, alpha=0.6, label='VAMOS drop', zorder=4)
ax1.set_xlabel('Temperature [°C]', fontsize=11)
ax1.set_ylabel('Altitude [m AGL]', fontsize=11)
ax1.set_title('Temperature\nT(h)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=8, loc='upper right')
ax1.grid(alpha=0.3); ax1.set_ylim(*H_LIM)

# Annotation: cold anomaly
if era5_ok:
    bias_era5 = np.mean(T_vd) - np.interp(np.mean(h_vd),
                era5_profiles['h_agl'], era5_profiles['T'])
    ax1.text(0.04, 0.04,
             f'VAMOS bias vs ERA5:\n{bias_era5:+.1f}°C',
             transform=ax1.transAxes, fontsize=8.5, family='monospace',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

# ── Panel 2: Pressure ─────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
ax2.plot(p_isa, h_isa_agl, 'k--', lw=2, label='ISA', zorder=1)
ax2.plot(pay['p'], pay['h_agl'], color=C_PAY, lw=2.5, marker='o', ms=3,
         label='Payerne', zorder=2)
if era5_ok:
    ax2.plot(era5_profiles['p'], era5_profiles['h_agl'],
             color=C_ERA5, lw=2.5, marker='s', ms=6,
             label='ERA5', zorder=3)
ax2.scatter(p_v[dm_v], h_vd, s=12, c=C_VAMOS, alpha=0.6, label='VAMOS drop', zorder=4)
ax2.invert_xaxis()
ax2.set_xlabel('Pressure [hPa]', fontsize=11)
ax2.set_ylabel('Altitude [m AGL]', fontsize=11)
ax2.set_title('Pressure\np(h)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=8, loc='upper right')
ax2.grid(alpha=0.3); ax2.set_ylim(*H_LIM)

# ── Panel 3: Relative Humidity ────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[2])
ax3.plot(pay['rh'], pay['h_agl'], color=C_PAY, lw=2.5, marker='o', ms=3,
         label='Payerne RH')
if era5_ok:
    ax3.plot(era5_profiles['rh'], era5_profiles['h_agl'],
             color=C_ERA5, lw=2.5, marker='s', ms=6,
             label='ERA5 RH')
ax3.axvline(100, color='gray', ls=':', lw=1.5, alpha=0.7, label='RH = 100%')
ax3.axvspan(95, 101, color='lightblue', alpha=0.2, label='Fog/cloud layer')
ax3.set_xlabel('Relative Humidity [%]', fontsize=11)
ax3.set_ylabel('Altitude [m AGL]', fontsize=11)
ax3.set_title('Relative Humidity\nRH(h)', fontsize=12, fontweight='bold')
ax3.legend(fontsize=8, loc='lower right')
ax3.grid(alpha=0.3); ax3.set_ylim(*H_LIM); ax3.set_xlim(0, 105)
ax3.text(0.04, 0.04,
         'RH = 100% up to ~900m\n→ fog/stratiform clouds',
         transform=ax3.transAxes, fontsize=8.5, color='steelblue',
         bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.85))

# ── Panel 4: Wind Speed ───────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[3])
ax4.plot(pay['wspd'], pay['h_agl'], color=C_PAY, lw=2.5, marker='o', ms=3,
         label='Payerne wind')
if era5_ok:
    ax4.plot(era5_profiles['wspd'], era5_profiles['h_agl'],
             color=C_ERA5, lw=2.5, marker='s', ms=6,
             label='ERA5 wind')
ax4.set_xlabel('Wind Speed [m/s]', fontsize=11)
ax4.set_ylabel('Altitude [m AGL]', fontsize=11)
ax4.set_title('Wind Speed\nwspd(h)', fontsize=12, fontweight='bold')
ax4.legend(fontsize=8, loc='upper left')
ax4.grid(alpha=0.3); ax4.set_ylim(*H_LIM)

era5_label = (f"ERA5 @ Dübendorf ({era5_profiles['time'][:13]})"
              if era5_ok else "ERA5 (not loaded)")

fig.suptitle(
    f'Atmospheric Vertical Profile — Dübendorf, 6 February 2026\n'
    f'VAMOS (blue) vs ISA (black dashed) vs Payerne WMO 06610 (orange) vs {era5_label} (pink)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
era5_plot_path = OUTPUT_DIR / 'fig_era5_vertical_profile.png'
plt.savefig(era5_plot_path, dpi=140, bbox_inches='tight')
plt.close()
print(f"  ✓ Saved: {era5_plot_path}")



In [ ]:
# ============================================================================
# STEP 6: SUMMARY TABLE
# ============================================================================

print("\n" + "="*65)
print("VERTICAL PROFILE SUMMARY")
print("="*65)
print(f"{'Dataset':<20} {'T at 400m':>10} {'p at 400m':>10} {'Source'}")
print("-"*65)

h_ref = 400
T_isa_400 = isa_T(h_ref + ELEV_ASL) - 273.15
p_isa_400 = isa_p(h_ref + ELEV_ASL)
T_vamos_400 = np.interp(h_ref, np.sort(h_vd), T_vd[np.argsort(h_vd)])
T_pay_400   = np.interp(h_ref, pay['h_agl'].values, pay['T'].values)
p_pay_400   = np.interp(h_ref, pay['h_agl'].values, pay['p'].values)

print(f"{'ISA':<20} {T_isa_400:>9.1f}°C {p_isa_400:>9.1f} hPa  theoretical")
print(f"{'VAMOS (measured)':<20} {T_vamos_400:>9.1f}°C {'—':>10}  in-situ CanSat")
print(f"{'Payerne':<20} {T_pay_400:>9.1f}°C {p_pay_400:>9.1f} hPa  radiosonde 140km SW")

if era5_ok:
    T_era5_400 = np.interp(h_ref, era5_profiles['h_agl'], era5_profiles['T'])
    p_era5_400 = np.interp(h_ref, era5_profiles['h_agl'], era5_profiles['p'])
    print(f"{'ERA5 (Dübendorf)':<20} {T_era5_400:>9.1f}°C {p_era5_400:>9.1f} hPa  reanalysis on-site")

print(f"\nVAMOS bias at 400m:")
print(f"  vs ISA:    {T_vamos_400-T_isa_400:+.1f}°C")
print(f"  vs Payerne:{T_vamos_400-T_pay_400:+.1f}°C")
if era5_ok:
    print(f"  vs ERA5:   {T_vamos_400-T_era5_400:+.1f}°C")
print("="*65)